In [ ]:
import pickle
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
DATA = Path('/kaggle/input/mirror-notes-temp')
OUT  = Path('/kaggle/working')

In [ ]:
MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print('ClinicalBERT loaded')

In [ ]:
def encode_notes(notes_pkl_path, cohort_pkl_path, out_embed_path, out_mean_path, batch_size=256):
    with open(notes_pkl_path, 'rb') as f:
        note_data = pickle.load(f)
    with open(cohort_pkl_path, 'rb') as f:
        cohort = pickle.load(f)

    hadm_ids  = np.array(note_data['hadm_ids'])
    notes_dict = note_data['notes']  # {hadm_id: text}
    N = len(hadm_ids)
    print(f'  Cohort: {N} admissions, {len(notes_dict)} with notes')

    CHUNK = 510  # 512 - 2 for [CLS]/[SEP]
    STRIDE = 382  # 512 - 128 overlap - 2

    # --- Pass 1: tokenize all notes, build chunk index ---
    print('  Tokenising...')
    all_input_ids = []   # flat list of chunk token tensors
    chunk_to_note = []   # which note index each chunk belongs to
    note_to_idx   = {}   # hadm_id -> position in hadm_ids array

    for pos, hid in enumerate(hadm_ids):
        note_to_idx[int(hid)] = pos

    has_note = np.zeros(N, dtype=bool)
    note_chunk_start = np.zeros(N, dtype=np.int32)  # first chunk index per note
    note_chunk_count = np.zeros(N, dtype=np.int32)  # number of chunks per note

    for hid, text in notes_dict.items():
        pos = note_to_idx.get(int(hid))
        if pos is None or not isinstance(text, str) or not text.strip():
            continue
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if not tokens:
            continue
        note_chunk_start[pos] = len(all_input_ids)
        n_chunks = 0
        for s in range(0, len(tokens), STRIDE):
            chunk = tokens[s:s + CHUNK]
            ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
            all_input_ids.append(ids)
            chunk_to_note.append(pos)
            n_chunks += 1
            if s + CHUNK >= len(tokens):
                break
        note_chunk_count[pos] = n_chunks
        has_note[pos] = True

    total_chunks = len(all_input_ids)
    print(f'  {total_chunks} total chunks from {has_note.sum()} notes')

    # --- Pass 2: encode all chunks in GPU batches ---
    print('  Encoding chunks on GPU...')
    chunk_embeddings = np.zeros((total_chunks, 768), dtype=np.float32)

    for start in range(0, total_chunks, batch_size):
        end = min(start + batch_size, total_chunks)
        batch = all_input_ids[start:end]
        max_len = max(len(x) for x in batch)
        input_ids = torch.zeros(len(batch), max_len, dtype=torch.long, device=device)
        attn_mask = torch.zeros(len(batch), max_len, dtype=torch.long, device=device)
        for i, ids in enumerate(batch):
            input_ids[i, :len(ids)] = torch.tensor(ids)
            attn_mask[i, :len(ids)] = 1
        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attn_mask)
            cls = out.last_hidden_state[:, 0, :].cpu().numpy()
        chunk_embeddings[start:end] = cls
        if start % (batch_size * 20) == 0:
            print(f'  {end}/{total_chunks} chunks ({100*end/total_chunks:.1f}%)')

    # --- Pass 3: mean-pool chunks back to note level ---
    print('  Mean-pooling chunks to notes...')
    embeddings = np.zeros((N, 768), dtype=np.float32)
    for pos in range(N):
        if not has_note[pos]:
            continue
        s = note_chunk_start[pos]
        c = note_chunk_count[pos]
        embeddings[pos] = chunk_embeddings[s:s+c].mean(axis=0)

    # --- Save embeddings ---
    embed_out = {'embeddings': embeddings, 'has_note': has_note,
                 'hadm_ids': hadm_ids, 'method': 'clinicalbert_chunk_pool'}
    with open(out_embed_path, 'wb') as f:
        pickle.dump(embed_out, f)
    print(f'  Saved embeddings -> {out_embed_path}')

    # --- Compute training-set mean (anisotropy correction) ---
    split = np.array(cohort['split'])
    train_mask = (split == 'train') & has_note
    global_mean = embeddings[train_mask].mean(axis=0)
    np.save(out_mean_path, global_mean)
    print(f'  Saved note mean   -> {out_mean_path}  (from {train_mask.sum()} train notes)')

    # quick anisotropy check
    E = embeddings[has_note]
    E_c = E - global_mean
    n = min(500, len(E))
    rng = np.random.default_rng(42)
    idx = rng.choice(len(E), n, replace=False)
    def mean_cos(M):
        M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-8)
        c = M @ M.T
        return c[np.triu_indices(n, k=1)].mean()
    print(f'  Cosine sim BEFORE centering: {mean_cos(E[idx]):.4f}')
    print(f'  Cosine sim AFTER  centering: {mean_cos(E_c[idx]):.4f}')
    print()

In [ ]:
# === mimic4 (ICD-9 only) — skip if files not uploaded ===
if (DATA / 'notes_text_mimic4.pkl').exists():
    print('=== mimic4 (ICD-9 only) ===')
    encode_notes(
        notes_pkl_path = DATA / 'notes_text_mimic4.pkl',
        cohort_pkl_path = DATA / 'cohort_mimic4.pkl',
        out_embed_path  = OUT  / 'note_embeddings_mimic4.pkl',
        out_mean_path   = OUT  / 'note_global_mean_mimic4.npy',
        batch_size=256
    )
else:
    print('SKIP: notes_text_mimic4.pkl not found — skipping mimic4')

In [ ]:
print('=== mimic4_sota (strict ICD-9, ~9K patients, SOTA-comparable) ===')
encode_notes(
    notes_pkl_path = DATA / 'notes_text_mimic4_sota.pkl',
    cohort_pkl_path = DATA / 'cohort_mimic4_sota.pkl',
    out_embed_path  = OUT  / 'note_embeddings_mimic4_sota.pkl',
    out_mean_path   = OUT  / 'note_global_mean_mimic4_sota.npy',
    batch_size=256
)

In [ ]:
# === mimic4_full (ICD-9 + ICD-10) — skip if files not uploaded ===
if (DATA / 'notes_text_mimic4_full.pkl').exists():
    print('=== mimic4_full (ICD-9 + ICD-10) ===')
    encode_notes(
        notes_pkl_path = DATA / 'notes_text_mimic4_full.pkl',
        cohort_pkl_path = DATA / 'cohort_mimic4_full.pkl',
        out_embed_path  = OUT  / 'note_embeddings_mimic4_full.pkl',
        out_mean_path   = OUT  / 'note_global_mean_mimic4_full.npy',
        batch_size=256
    )
else:
    print('SKIP: notes_text_mimic4_full.pkl not found — skipping mimic4_full')

In [ ]:
import os
for f in sorted(Path('/kaggle/working').glob('note_*')):
    print(f.name, f'{f.stat().st_size/1e6:.1f} MB')